In [2]:
#Wikipedia tool creation
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
wikipedia = WikipediaAPIWrapper(top_k_results=1)
tool = WikipediaQueryRun(api_wrapper=wikipedia)

In [3]:
tool.name

'wikipedia'

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("LANGCHAIN_KEY")

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 4


In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

db=Chroma.from_documents(documents,GoogleGenerativeAIEmbeddings(model='gemini-embedding-001'))

retriever=db.as_retriever()



In [7]:
#Langsmith tool creation
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool=create_retriever_tool(
    name="Langsmith",
    description="Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
    retriever=retriever
)

In [8]:
retriever_tool.name

'Langsmith'

In [9]:
#arxiv tool creation
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
arxiv = ArxivAPIWrapper(top_k_results=1)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv)

In [10]:
tools=[tool,retriever_tool,arxiv_tool]

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash",temperature=0)

In [12]:
### Agents
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant that answers clearly."
)

In [13]:
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about Langsmith"}
    ]
})